# 01 · Prepare TWOSIDES and OFFSIDES (MedSafe-AI · Component 3)

**What this notebook does**
1. Reads the big `TWOSIDES.csv` file **in small pieces** (it has about 43 million rows, so it can not be loaded at once).
2. Removes bad lines (the file has a header line repeated in the middle) and keeps only **strong signals**.
3. Checks the data quality.
4. Builds the **drug pair × side effect** table (this is what the models learn from).
5. Builds a list of known single-drug side effects from `OFFSIDES.csv`.

**Time:** about 3–6 minutes. Your original files are never changed. Everything new is saved in the `work` folder.

> Remember: TWOSIDES comes from adverse-event **reports**. A side effect that is *not* listed does **not** mean it is safe, and a listed one is a statistical signal, not proof of cause.

In [ ]:
import sys, importlib.util, subprocess
from pathlib import Path

HERE = Path.cwd()                                   # the folder of this notebook (component3)
if not (HERE / "c3_utils.py").exists():             # Jupyter was started from another folder
    HERE = Path(r"D:\med\data set\component3")
sys.path.insert(0, str(HERE))

def need(*pairs):
    """Install the packages that are missing. pairs = (import name, pip name)."""
    missing = [pip for mod, pip in pairs if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", missing)
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        except Exception as e:
            print("!! Could not install", missing, "->", e)

need(("pandas", "pandas"), ("numpy", "numpy"), ("matplotlib", "matplotlib"))

import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import c3_utils as c3

DATA_DIR = c3.find_data_dir()
WORK = HERE / "work"
WORK.mkdir(exist_ok=True)
TWOSIDES = c3.find_file(DATA_DIR, "TWOSIDES.csv")
OFFSIDES = c3.find_file(DATA_DIR, "OFFSIDES.csv")

print("Data folder :", DATA_DIR)
print("TWOSIDES    :", TWOSIDES, f"({TWOSIDES.stat().st_size/1e9:.2f} GB)")
print("OFFSIDES    :", OFFSIDES, f"({OFFSIDES.stat().st_size/1e6:.0f} MB)")
print("Work folder :", WORK)

## Step 1 · Settings
You can change these later and run the notebook again.

| Setting | Meaning |
|---|---|
| `PRR_MIN` | PRR ≥ 2 means the drug pair makes the side effect at least 2× more likely to be reported than expected. |
| `A_MIN` | Minimum number of reports that mention this drug pair **and** this side effect (small numbers are noisy). |
| `TOP_K` | How many of the most common side effects to predict. |
| `EXTRA_HIGH_MAX` | Plus up to this many rarer but **serious** (High severity) side effects, so dangerous ones are not lost. |

In [ ]:
PRR_MIN = 2.0
A_MIN = 3
TOP_K = 50
EXTRA_HIGH_MIN_PAIRS = 500     # a rare serious effect must appear in at least this many pairs
EXTRA_HIGH_MAX = 20

## Step 2 · Look at the first rows

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 20)
print("TWOSIDES (first 3 rows)")
display(pd.read_csv(TWOSIDES, nrows=3))
print("OFFSIDES (first 3 rows)")
display(pd.read_csv(OFFSIDES, nrows=3))

## Step 3 · Read TWOSIDES in pieces and keep the strong signals
This is the slow step (about 2–5 minutes). The result is saved, so next time it loads in seconds.

In [ ]:
SIG_FILE = WORK / "signals.npz"
META_FILE = WORK / "signals_meta.json"

if SIG_FILE.exists() and META_FILE.exists():
    z = np.load(SIG_FILE)
    sig = {k: z[k] for k in z.files}
    meta = json.load(open(META_FILE))
    sig["drug_names"] = {int(k): v for k, v in meta["drug_names"].items()}
    sig["effect_names"] = {int(k): v for k, v in meta["effect_names"].items()}
    sig["stats"] = meta["stats"]
    print("Loaded saved signals:", SIG_FILE.name)
else:
    t0 = time.time()
    sig = c3.load_twosides_signals(TWOSIDES, prr_min=PRR_MIN, a_min=A_MIN, offsides_path=OFFSIDES)
    np.savez_compressed(SIG_FILE, **{k: sig[k] for k in ["lo", "hi", "d1", "d2", "eff", "A", "PRR"]})
    json.dump({"drug_names": {str(k): v for k, v in sig["drug_names"].items()},
               "effect_names": {str(k): v for k, v in sig["effect_names"].items()},
               "stats": sig["stats"]}, open(META_FILE, "w"))
    print(f"Done in {time.time()-t0:.0f}s and saved to {SIG_FILE.name}")

## Step 4 · Data quality checks (TWOSIDES)
Look at these numbers carefully. They are your *evidence* that the data is complete and clean.

In [ ]:
st = sig["stats"]
lo, hi, eff = sig["lo"], sig["hi"], sig["eff"]

n_ordered   = len(np.unique(sig["d1"] * c3.KEY_BASE + sig["d2"]))
n_unordered = len(np.unique(lo * c3.KEY_BASE + hi))
pe = pd.DataFrame({"k": lo * c3.KEY_BASE + hi, "e": eff})

quality = pd.DataFrame({"value": {
    "valid rows read from file": f"{st['rows_read']:,}",
    "bad lines removed (e.g. repeated header)": st["bad_rows"],
    f"rows with PRR >= {st['prr_min']}": f"{st['rows_prr_ok']:,}",
    f"rows kept (PRR >= {st['prr_min']} and A >= {st['a_min']})": f"{st['rows_kept']:,}",
    "drug pairs (order ignored)": f"{n_unordered:,}",
    "pairs stored in both orders A-B and B-A": f"{n_ordered - n_unordered:,}",
    "pairs where both drugs are the same": int((lo == hi).sum()),
    "duplicate (pair, side effect) rows": int(pe.duplicated().sum()),
    "different drugs": f"{len(np.unique(np.concatenate([lo, hi]))):,}",
    "different side effects": f"{len(np.unique(eff)):,}",
}})
display(quality)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
per_pair = pe.drop_duplicates().groupby("k").size()
ax[0].hist(per_pair.clip(upper=150), bins=50, color="#3b6ea5")
ax[0].set_title("Side effects per drug pair"); ax[0].set_xlabel("side effects (cut at 150)")
ax[1].hist(sig["PRR"].clip(max=30), bins=60, color="#c0504d")
ax[1].set_title("PRR of kept signals"); ax[1].set_xlabel("PRR (cut at 30)")
plt.tight_layout(); plt.show()
print("A (reports with drug pair + effect): median", int(np.median(sig["A"])), " 90th percentile", int(np.percentile(sig["A"], 90)))

## Step 5 · Build the drug pair × side effect table
* One row = one drug pair (A+B is the same as B+A).
* One column = one side effect. `1` means TWOSIDES reports a strong signal for that pair.
* We keep the `TOP_K` most common side effects, plus a few rarer serious ones.
* **Severity (Low / Medium / High)** here comes from the *placeholder* rules in `severity.py`. Agree the real source with your supervisor before you report severity results.

In [ ]:
pairs, Y, effects = c3.build_pairs_and_labels(
    sig, top_k=TOP_K, extra_high_min_pairs=EXTRA_HIGH_MIN_PAIRS, extra_high_max=EXTRA_HIGH_MAX)

# drug names for the drugs that are in the pairs
ids = np.unique(np.concatenate([pairs["drug_a_id"], pairs["drug_b_id"]]))
drugs = pd.DataFrame({"drug_id": ids, "drug_name": [sig["drug_names"].get(int(i), str(i)) for i in ids]})

pairs.to_csv(WORK / "pairs_raw.csv", index=False)
np.save(WORK / "Y_raw.npy", Y)
effects.to_csv(WORK / "effects.csv", index=False)
drugs.to_csv(WORK / "drugs_raw.csv", index=False)

print(f"pairs: {len(pairs):,}   drugs: {len(drugs):,}   side effects (labels): {Y.shape[1]}")
print(f"average side effects per pair: {Y.sum(1).mean():.1f}   positive density: {100*Y.mean():.2f}%")
print("severity of chosen side effects:", effects["severity"].value_counts().to_dict())
display(effects.head(15))

## Step 6 · Known single-drug side effects from OFFSIDES
OFFSIDES lists side effects of **one drug** (not a pair). We keep the 15 most frequently reported strong signals per drug.
These are the "known side effects" that the final application will show for each medicine.

In [ ]:
stats_off = {}
header = pd.read_csv(OFFSIDES, nrows=0).columns.tolist()
c_id = c3._column(header, "drug_rxn")
cols = [c_id, "drug_concept_name", "condition_concept_name", "A", "PRR", "mean_reporting_frequency"]
num = [c_id, "A", "PRR", "mean_reporting_frequency"]
kept = []
for ch in c3.read_clean_chunks(OFFSIDES, cols, num, stats_off):
    kept.append(ch[(ch["PRR"] >= 2) & (ch["A"] >= 5)])
off = pd.concat(kept, ignore_index=True).rename(columns={
    c_id: "drug_id", "drug_concept_name": "drug_name", "condition_concept_name": "effect",
    "mean_reporting_frequency": "frequency"})
off["drug_id"] = off["drug_id"].astype("int64")
off = off.sort_values(["drug_id", "frequency"], ascending=[True, False]).groupby("drug_id").head(15)
off["severity"] = [c3.severity_of(e) or "Unknown" for e in off["effect"]]
off.to_csv(WORK / "offsides_top_effects.csv", index=False)

print(f"OFFSIDES valid rows: {stats_off['rows_read']:,}   bad lines removed: {stats_off['bad_rows']}")
print(f"drugs with known side effects: {off['drug_id'].nunique():,}   rows saved: {len(off):,}")
shared = len(set(drugs['drug_id']) & set(off['drug_id']))
print(f"drugs of our pairs that also have OFFSIDES info: {shared:,} of {len(drugs):,}")
display(off[off['drug_name'].str.lower() == 'warfarin'].head(8))

## Done ✔
Saved in the `work` folder: `signals.npz`, `pairs_raw.csv`, `Y_raw.npy`, `effects.csv`, `drugs_raw.csv`, `offsides_top_effects.csv`.

**Next:** open `02_drug_details_and_features.ipynb`.